# Recitation 0: Datasets, Part 3 - Making the Dataset Fast

Part 1 covered the `Dataset` interface. Part 2 covered the HW1P2 MFCC data and context windows.
This notebook is about the gap between a `Dataset` that is **correct** and one that is **fast enough
to keep your GPU busy**.

The rule that organises everything below:

> **epoch time = max(GPU compute time, data pipeline time)**

Both run concurrently once you have workers, so the slower one sets your epoch time. Every technique
here moves work out of the per-sample hot path. Every one of them costs you something, and each
section states the cost explicitly.

## 0. Setup

In [ ]:
import os, glob, time, hashlib, json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, BatchSampler, RandomSampler

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# On Colab, unzip first. The -o flag overwrites, so re-running is safe.
# !unzip -qo /content/Recitation_0.15_data.zip -d /content

# Locate the data directory by structure, not by name. Identical to Parts 1 and 2.
ROOT = None
for cand in glob.glob("/content/**/mfcc", recursive=True) or glob.glob("**/mfcc", recursive=True):
    ROOT = os.path.dirname(cand)
    break
assert ROOT is not None, "Could not find a directory containing mfcc/ - check where you unzipped."

MFCC_FILES = sorted(glob.glob(os.path.join(ROOT, "mfcc", "*.npy")))
TRAN_FILES = sorted(glob.glob(os.path.join(ROOT, "transcript", "*.npy")))
assert len(MFCC_FILES) == len(TRAN_FILES) and len(MFCC_FILES) > 0

print(f"root         : {ROOT}")
print(f"utterances   : {len(MFCC_FILES)}")
x0 = np.load(MFCC_FILES[0]); y0 = np.load(TRAN_FILES[0], allow_pickle=True)
print(f"x[0].shape   : {x0.shape}  dtype {x0.dtype}")
print(f"y[0].shape   : {y0.shape}  dtype {y0.dtype}   first tokens {y0[:3]}")

In [ ]:
# The phoneme vocabulary. Build the string -> int lookup ONCE, here, not inside __getitem__.
PHONEMES = sorted({s for f in TRAN_FILES for s in np.load(f, allow_pickle=True)[1:-1].tolist()})
LUT = {p: i for i, p in enumerate(PHONEMES)}
CONTEXT = 20
NFEAT = 28
print(f"{len(PHONEMES)} phonemes | input_dim = (2*{CONTEXT}+1)*{NFEAT} = {(2*CONTEXT+1)*NFEAT}")

## 1. The baseline: a correct, slow Dataset

This is the implementation almost everyone writes first. Nothing about it is wrong. It is the
reference we will diff every optimised version against.

Three costs live in `__getitem__`, and each is paid once per sample per epoch:

1. `np.load` - an `open()`, a header parse and a read, per access
2. `np.pad` - a fresh allocation of the whole utterance, per access
3. `torch.tensor(...)` - a copy, and a silent dtype promotion if the input is a Python float

In [ ]:
class NaiveDataset(Dataset):
    def __init__(self, mfcc_files, tran_files, context=CONTEXT):
        self.files = mfcc_files
        self.context = context
        # (utterance_index, frame_index) for every frame in the split
        self.index_map, self.labels = [], []
        for u, tf in enumerate(tran_files):
            lab = np.load(tf, allow_pickle=True)[1:-1]      # strip [SOS] and [EOS]
            for t in range(len(lab)):
                self.index_map.append((u, t))
                self.labels.append(LUT[lab[t]])

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, i):
        u, t = self.index_map[i]
        arr = np.load(self.files[u])                        # (1) disk read, every access
        pad = np.pad(arr, ((self.context, self.context), (0, 0)))   # (2) allocation, every access
        w = pad[t : t + 2 * self.context + 1]
        return torch.tensor(w.reshape(-1), dtype=torch.float32), self.labels[i]   # (3) copy

naive = NaiveDataset(MFCC_FILES, TRAN_FILES)
xi, yi = naive[100]
print(len(naive), xi.shape, xi.dtype, yi)

## 2. Rule 0: measure before you optimise

Two numbers tell you whether the loader is your problem at all.

- **Data-only epoch time**: iterate the loader with no model attached.
- **GPU utilisation**: `watch -n0.5 nvidia-smi` in a second terminal, or `!nvidia-smi` in a loop.

If utilisation sits below ~50 percent you are input-bound and everything below is worth doing.
If it is pinned near 100 percent, your loader is already keeping up: stop here and go tune the model.

In [ ]:
def data_only_epoch(loader, max_batches=None):
    t0 = time.perf_counter()
    seen = 0
    for i, (x, y) in enumerate(loader):
        seen += x.shape[0]
        if max_batches is not None and i + 1 >= max_batches:
            break
    dt = time.perf_counter() - t0
    return dt, seen / dt

loader = DataLoader(naive, batch_size=256, shuffle=True, num_workers=0)
dt, rate = data_only_epoch(loader, max_batches=20)
print(f"naive: {dt:.2f}s for 20 batches | {rate:,.0f} samples/s")

## 3. Technique 1 - preload into RAM in `__init__`

Move the disk read out of the hot path. You pay it once per run instead of once per access.

**Cost.** Construction gets slow, and the whole split must fit in memory. `train-clean-100` as
float32 is about 3.1 GB, which fits on Colab. If your data does not fit, skip to the memory map
in section 7.

In [ ]:
class PreloadDataset(Dataset):
    def __init__(self, mfcc_files, tran_files, context=CONTEXT):
        self.context = context
        self.utts = [np.load(f).astype(np.float32) for f in mfcc_files]   # ONCE
        self.labels, self.index_map = [], []
        for u, tf in enumerate(tran_files):
            lab = np.load(tf, allow_pickle=True)[1:-1]
            for t in range(len(lab)):
                self.index_map.append((u, t))
                self.labels.append(LUT[lab[t]])
        self.labels = np.asarray(self.labels, dtype=np.int64)

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, i):
        u, t = self.index_map[i]
        arr = self.utts[u]
        lo, hi = t - self.context, t + self.context + 1
        w = arr[max(0, lo) : min(len(arr), hi)]
        if w.shape[0] < 2 * self.context + 1:                # still padding per access
            before = max(0, -lo)
            after = 2 * self.context + 1 - w.shape[0] - before
            w = np.pad(w, ((before, after), (0, 0)))
        return torch.from_numpy(np.ascontiguousarray(w.reshape(-1))), self.labels[i]

preload = PreloadDataset(MFCC_FILES, TRAN_FILES)
print(len(preload), preload[100][0].shape)

## 4. Technique 2 - flatten, and do index arithmetic

The training sample is a **frame**, not an utterance. So stop maintaining utterance structure at
access time: concatenate every utterance into one contiguous array and let the frame index be the
only index you need.

**Cost.** `np.concatenate` transiently needs double the memory, so free the per-utterance list right
after. Utterance boundaries disappear, so a window can straddle two speakers. For frame-level
classification that is accepted; for anything sequential it is a bug.

In [ ]:
utts = [np.load(f).astype(np.float32) for f in MFCC_FILES]
flat = np.concatenate(utts, axis=0)                 # (N, 28)
del utts                                            # release the transient copy

labels = np.concatenate([
    np.array([LUT[s] for s in np.load(f, allow_pickle=True)[1:-1].tolist()], dtype=np.int64)
    for f in TRAN_FILES
])
assert len(flat) == len(labels), "frames and labels must line up after stripping SOS/EOS"
print(f"flat {flat.shape} {flat.dtype} | {flat.nbytes/1e6:.2f} MB | labels {labels.shape}")

## 5. Technique 3 - pad once, then return views

Pad the flat array a single time in `__init__`. After that, frame `i` is exactly
`padded[i : i + 2*context + 1]`, which numpy returns as a **view**: no allocation, no memcpy.
The boundary special case disappears entirely.

**Cost.** The padding costs `2 * context` extra rows, negligible here. `reshape(-1)` is free only
because a row-slice of a C-contiguous array is itself contiguous. Store your features transposed as
`(28, T)` and numpy will silently copy on every access.

In [ ]:
padded = np.pad(flat, ((CONTEXT, CONTEXT), (0, 0)))

w = padded[100 : 100 + 2 * CONTEXT + 1]
print("is the window a view, not a copy? ", np.shares_memory(w, padded))
print("is it contiguous (so reshape is free)? ", w.flags["C_CONTIGUOUS"])

class FlatDataset(Dataset):
    def __init__(self, flat, labels, context=CONTEXT):
        self.padded = np.pad(flat, ((context, context), (0, 0)))
        self.labels = labels
        self.context = context
        self.width = 2 * context + 1

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        w = self.padded[i : i + self.width]
        return torch.from_numpy(w.reshape(-1)), self.labels[i]

fast = FlatDataset(flat, labels)
print(len(fast), fast[100][0].shape, fast[100][0].dtype)

## 6. Technique 4 - dtype discipline

Half the bytes means half the memory traffic. Three rules:

- features in `float32`, never `float64`. GPUs are built for float32 and a float64 array forces a
  cast on every batch.
- labels as integers, mapped once. `'<U5'` numpy strings are 20 bytes each and cost a hash on every
  lookup if you resolve them inside `__getitem__`.
- `torch.from_numpy` **shares** memory; `torch.tensor` **copies**. Know which one you called.

**Cost.** `CrossEntropyLoss` needs int64 targets, so if you store int8 you cast back at batch level.
That is one cheap op per batch rather than one per sample.

In [ ]:
y_str = np.concatenate([np.load(f, allow_pickle=True)[1:-1] for f in TRAN_FILES])
print(f"'<U5' strings : {y_str.nbytes/1e6:7.3f} MB")
print(f"int64         : {labels.nbytes/1e6:7.3f} MB")
print(f"int8          : {labels.astype(np.int8).nbytes/1e6:7.3f} MB")

t = torch.from_numpy(padded[100:141].reshape(-1))
print("from_numpy shares memory with the numpy array:", t.data_ptr() == padded[100:141].__array_interface__["data"][0])

## 7. Technique 5 - caching, three levels

Ordered by what each level survives.

| Level | Mechanism | Survives | Main cost |
|---|---|---|---|
| L1 | in-process dict, `functools.lru_cache` | the epoch | one copy **per worker** |
| L2 | preprocessed array written with `np.save` | the process | goes stale silently |
| L3 | `np.load(..., mmap_mode='r')` | running out of RAM | first-epoch page faults |

**Cache invalidation is your job.** Key the cache filename on a hash of the preprocessing config, or
you will happily load a cache built with `context=10` while running `context=20`.

In [ ]:
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_key(**cfg):
    blob = json.dumps(cfg, sort_keys=True).encode()
    return hashlib.sha1(blob).hexdigest()[:10]

CFG = dict(split="demo", context=CONTEXT, nfeat=NFEAT, dtype="float32", n_files=len(MFCC_FILES))
KEY = cache_key(**CFG)
CACHE_X = os.path.join(CACHE_DIR, f"flat_{KEY}.npy")
CACHE_Y = os.path.join(CACHE_DIR, f"labels_{KEY}.npy")
print("cache key:", KEY)

def build_flat():
    us = [np.load(f).astype(np.float32) for f in MFCC_FILES]
    return np.concatenate(us, axis=0)

# L2: write once, reuse on every later run / kernel restart
t0 = time.perf_counter()
if not os.path.exists(CACHE_X):
    np.save(CACHE_X, build_flat())
    np.save(CACHE_Y, labels)
    print(f"built and wrote cache in {time.perf_counter()-t0:.3f}s")

t0 = time.perf_counter(); _ = build_flat();                       t_src   = time.perf_counter() - t0
t0 = time.perf_counter(); _ = np.load(CACHE_X);                   t_cache = time.perf_counter() - t0
t0 = time.perf_counter(); mm = np.load(CACHE_X, mmap_mode="r");   t_mmap  = time.perf_counter() - t0

print(f"re-read source files : {t_src*1e3:8.2f} ms   1.0x")
print(f"load cached .npy     : {t_cache*1e3:8.2f} ms   {t_src/t_cache:.1f}x")
print(f"mmap cached .npy     : {t_mmap*1e3:8.2f} ms   {t_src/t_mmap:.1f}x   (lazy: pages load on demand)")

On Colab, write L2 into your mounted Drive. It survives a runtime disconnect and saves you the
preprocessing pass at the start of every session.

Two caching mistakes that cost people a homework:

- **Caching augmented samples.** Augmentation exists to be random. Cache the raw array; augment in
  `collate_fn` or on the GPU.
- **Sharing a cache across splits.** Fit normalisation statistics on train, then reuse those exact
  statistics for val and test. A cache built from dev statistics leaks.

## 8. Technique 6 - batched fetch and vectorised collate

`__getitem__` runs `batch_size` times per batch. At `batch_size=1024` that is 1024 interpreter
round-trips and 1024 small allocations for one batch. PyTorch 2.x lets a `Dataset` expose
`__getitems__` (note the plural) to fetch a whole batch in one call, so you can replace the loop
with a single numpy gather.

**Cost.** The gather does copy, unavoidably, but it is one vectorised copy instead of 1024 small
ones. Per-sample random augmentation gets awkward: you now augment the batch as a unit.

In [ ]:
class BatchedFlatDataset(Dataset):
    def __init__(self, flat, labels, context=CONTEXT):
        self.padded = np.pad(flat, ((context, context), (0, 0)))
        self.labels = labels
        self.width = 2 * context + 1
        self.window = np.arange(self.width)          # precomputed offsets

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):                        # still needed as a fallback
        return torch.from_numpy(self.padded[i : i + self.width].reshape(-1)), self.labels[i]

    def __getitems__(self, idxs):                    # the fast path
        idxs = np.asarray(idxs)
        rows = idxs[:, None] + self.window[None, :]  # (B, width)
        x = self.padded[rows].reshape(len(idxs), -1) # ONE gather
        return torch.from_numpy(x), torch.from_numpy(self.labels[idxs])

batched = BatchedFlatDataset(flat, labels)

def identity_collate(b):     # __getitems__ already returned a batch
    return b

B = 1024
bidx = np.random.randint(0, len(fast), size=B)

def loop_collate():
    return torch.stack([torch.from_numpy(fast.padded[i:i+fast.width].reshape(-1)) for i in bidx])

def vectorised():
    return batched.__getitems__(bidx)[0]

for fn, name in [(loop_collate, "python loop + torch.stack"), (vectorised, "vectorised gather     ")]:
    fn(); t0 = time.perf_counter()
    for _ in range(10): fn()
    print(f"{name}: {(time.perf_counter()-t0)/10*1e3:7.3f} ms per batch of {B}")

## 9. Technique 7 - the DataLoader arguments that matter

```python
DataLoader(ds, batch_size=1024, shuffle=True,
           num_workers=4,             # overlap loading with compute
           pin_memory=True,           # page-locked -> async host to device copy
           persistent_workers=True,   # do not respawn workers every epoch
           prefetch_factor=2)         # batches queued per worker
```

**Costs.** Each worker is a separate process holding its own copy of the `Dataset`, so RAM scales
with `num_workers`. On a 2-core Colab VM, `num_workers=8` is *slower* than `num_workers=2` because
the workers thrash. `pin_memory` holds non-pageable memory and on a small VM can trigger the OOM
killer.

Measure. Do not copy a number from a Piazza post.

In [ ]:
def sweep(dataset, workers=(0, 2, 4), batch_size=1024, batches=15, collate=None):
    for w in workers:
        kw = dict(batch_size=batch_size, shuffle=True, num_workers=w,
                  pin_memory=torch.cuda.is_available())
        if w > 0:
            kw["persistent_workers"] = True
        if collate is not None:
            kw["collate_fn"] = collate
        dl = DataLoader(dataset, **kw)
        it = iter(dl); next(it)                       # absorb worker startup
        t0 = time.perf_counter(); seen = 0
        for i, (x, y) in enumerate(it):
            seen += x.shape[0]
            if i + 1 >= batches: break
        dt = time.perf_counter() - t0
        print(f"num_workers={w}: {seen/dt:10,.0f} samples/s")
        del dl

print("FlatDataset (per-sample __getitem__)")
sweep(fast)
print("\nBatchedFlatDataset (__getitems__)")
sweep(batched, collate=identity_collate)

## 10. Technique 8 - two traps that look like memory bugs

**Trap A - copy-on-write under `fork`.** Worker processes share memory with the parent until a page
is written. CPython writes to every object it touches, because it updates the refcount. So a Python
`list` of 36,000 strings gets gradually copied into *every* worker. A numpy array of the same data
does not, because it is one object holding one buffer.

```python
self.paths = np.array(paths)        # not a Python list of str
```

Symptom: "my Colab runs out of RAM at epoch 6." Only bites with `num_workers > 0`.

**Trap B - preprocessing per sample on the CPU.** Normalisation applied inside `__getitem__` runs
`batch_size` times on the CPU. Applied after `.to(device)` it is one kernel on the whole batch.

```python
x = (x.to(device, non_blocking=True) - mean) / std
```

**Cost.** Trap B moves work onto the GPU, which only helps while the GPU has headroom. Re-measure
after you move it.

## 11. Stay correct: diff against the naive version

Every technique above can silently shift your labels. Fancy indexing with a wrong offset produces
tensors of exactly the right shape and dtype, and the model trains, badly.

Keep the naive implementation in the file and assert against it. Four lines.

In [ ]:
for i in [0, 1, 37, len(fast) // 2, len(fast) - 1]:
    xf, yf = fast[i]
    xn, yn = naive[i]
    assert torch.allclose(xf, xn), f"features differ at index {i}"
    assert int(yf) == int(yn),     f"labels differ at index {i}"

xb, yb = batched.__getitems__(np.array([0, 1, 37, len(fast) // 2, len(fast) - 1]))
assert torch.allclose(xb[2], fast[37][0]), "batched fetch disagrees with per-sample fetch"
print("all implementations agree")

## 12. The summary benchmark

Re-run this on your own runtime. The ratios are what transfer; the absolute numbers are not.

In [ ]:
def bench(fn, n=20):
    fn()
    t0 = time.perf_counter()
    for _ in range(n): fn()
    return (time.perf_counter() - t0) / n

probe = np.random.randint(0, len(fast), size=64)

def A():  [naive[i]   for i in probe]
def Bf(): [preload[i] for i in probe]
def Cf(): [fast[i]    for i in probe]

tA, tB, tC = bench(A, 3), bench(Bf, 10), bench(Cf, 20)
print(f"{'per-sample __getitem__ (64 samples)':45s}{'ms':>10s}{'speedup':>10s}")
print(f"{'  A  re-load .npy from disk':45s}{tA*1e3:10.3f}{1.0:9.1f}x")
print(f"{'  B  preloaded into RAM':45s}{tB*1e3:10.3f}{tA/tB:9.1f}x")
print(f"{'  C  flat, padded once, views':45s}{tC*1e3:10.3f}{tA/tC:9.1f}x")

tD, tE = bench(loop_collate, 10), bench(vectorised, 10)
print(f"\n{'batch assembly (1024 samples)':45s}{'ms':>10s}{'speedup':>10s}")
print(f"{'  D  python loop + torch.stack':45s}{tD*1e3:10.3f}{1.0:9.1f}x")
print(f"{'  E  vectorised gather':45s}{tE*1e3:10.3f}{tD/tE:9.1f}x")

print(f"\nExtrapolated to train-clean-100 (~28M frames, context={CONTEXT}):")
print(f"  flat float32 array          : {28e6*NFEAT*4/1e9:6.2f} GB   (fits in Colab RAM)")
print(f"  materialised windows        : {28e6*(2*CONTEXT+1)*NFEAT*4/1e9:6.0f} GB   (never do this)")

## Which technique, when

| Question | If yes |
|---|---|
| Does the split fit in RAM? | Preload and flatten (1, 2, 3). If not, memory-map the cache (5, L3). |
| Is GPU utilisation below 50 percent? | Keep going. If not, you are compute-bound: stop and tune the model. |
| Is *construction* the slow part? | Cache the preprocessed array to disk (5, L2), keyed on your config. |
| Is per-sample overhead the slow part? | Batched fetch (6). |
| Are you using augmentation? | Cache the raw array, not the augmented one. |
| Still slow, and RAM keeps climbing? | Trap A: Python containers under `fork`. |

## Further reading

- PyTorch data tutorial: <https://pytorch.org/tutorials/beginner/basics/data_tutorial.html>
- `DataLoader` reference, especially the multi-process and memory-pinning notes: <https://pytorch.org/docs/stable/data.html>
- Profiler recipe: <https://pytorch.org/tutorials/recipes/recipes/profiler_recipe.html>
- numpy copies vs views: <https://numpy.org/doc/stable/user/basics.copies.html>